# Generate Token Graphs for Additional Structured Numerical Inputs

This Generate notebook creates the processed Llama-3.2-3B last-layer attention-graph and hidden-state archives consumed by `Visualize-token-graphs-additional-numerical-inputs.ipynb`. The companion Visualize notebook generates the graph and trajectory figures documented under **Attention-Induced Token Graphs for Additional Structured Numerical Inputs** in the appendix.

Archives cover five input families—a sine wave, Gaussian sequences with one or two regime switches, and bounded random walks with and without drift—at 10%, 30%, 50%, 70%, and 90% of a 1,000-token reference context. They store attention-derived graph weights, Fruchterman–Reingold layout positions, leading uncentered hidden-state SVD coordinates, and original state trajectories. These inputs extend the graph-organization comparisons beyond the main dynamical-system trajectories.

Set imports, constants, relative cache paths, retained inputs, and random seeds.


In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import random
import re
from pathlib import Path

import networkx as nx
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model, reference context, and relative cache path
MODEL_NAME = 'meta-llama/Llama-3.2-3B'
REFERENCE_CONTEXT_TOKENS = 1000
CONTEXT_FRACS = (0.10, 0.30, 0.50, 0.70, 0.90)  # Fractions of the 1,000-token reference context
STATE_COUNT = REFERENCE_CONTEXT_TOKENS // 2
PROCESSED_DATA_DIR = Path('processed_last_layer_additional_inputs_data')

# Input families
DATA_ORDER = [
    'sin(x)',
    'Gaussian switch @50%',
    'Gaussian switches @35%,65%',
    'Random walk',
    'Random walk with drift',
]

# Saved-array schema consumed by the companion Visualize notebook
GRAPH_DATA_ARRAY_KEYS = (
    'Wbar',
    'layout_positions',
    'svd_projection_coords',
    'is_delimiter',
    'state_values',
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Build the sine-wave, switching-Gaussian, and bounded-random-walk input sequences.


In [ ]:
# Integer encoding and reproducible sampling
input_integer_min = 50
input_integer_max = 950
rng = np.random.RandomState(SEED)


def discretize_to_integer_range(values, low=input_integer_min, high=input_integer_max):
    vals = np.asarray(values, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(vals.shape, int(round(0.5 * (low + high))), dtype=int)
    scaled = low + (vals - vmin) / (vmax - vmin) * (high - low)
    return np.round(scaled).astype(int)


def bounded_integer_sequence(values, low=150, high=850):
    vals = np.asarray(values, dtype=float)
    return np.clip(np.rint(vals), int(low), int(high)).astype(int)


def regime_switch_gaussian_sequence(means, switch_fracs, n, variance=10.0):
    switch_points = [0]
    switch_points.extend(int(round(float(frac) * n)) for frac in switch_fracs)
    switch_points.append(int(n))

    seq = np.empty(n, dtype=float)
    std = np.sqrt(float(variance))
    for mean, start, end in zip(means, switch_points[:-1], switch_points[1:]):
        seq[start:end] = rng.normal(loc=float(mean), scale=std, size=max(0, end - start))
    return seq


def bounded_random_walk_sequence(n, low=150.0, high=850.0, start=500.0, step_std=18.0, drift=0.0):
    seq = np.empty(n, dtype=float)
    seq[0] = float(start)
    low = float(low)
    high = float(high)
    for i in range(1, n):
        proposal = seq[i - 1] + float(drift) + rng.normal(loc=0.0, scale=float(step_std))
        # Reflect boundary crossings to keep the walk within the specified range.
        while proposal < low or proposal > high:
            if proposal < low:
                proposal = low + (low - proposal)
            if proposal > high:
                proposal = high - (proposal - high)
        seq[i] = proposal
    return seq


# Raw state trajectories and their integer representations for model input
oscillation_t = np.linspace(0.0, 6.0 * np.pi, STATE_COUNT)
sin_x_sequence = np.sin(oscillation_t)
sin_x_integers = discretize_to_integer_range(sin_x_sequence)

regime_switch_gaussian_50_sequence = regime_switch_gaussian_sequence(
    means=[300.0, 700.0],
    switch_fracs=[0.50],
    n=STATE_COUNT,
    variance=10.0,
)
regime_switch_gaussian_50_integers = bounded_integer_sequence(regime_switch_gaussian_50_sequence)

regime_switch_gaussian_35_65_sequence = regime_switch_gaussian_sequence(
    means=[300.0, 700.0, 300.0],
    switch_fracs=[0.35, 0.65],
    n=STATE_COUNT,
    variance=10.0,
)
regime_switch_gaussian_35_65_integers = bounded_integer_sequence(regime_switch_gaussian_35_65_sequence)

bounded_random_walk_values = bounded_random_walk_sequence(
    STATE_COUNT,
    low=150.0,
    high=850.0,
    start=500.0,
    step_std=18.0,
)
bounded_random_walk_integers = bounded_integer_sequence(bounded_random_walk_values)

bounded_random_walk_drift_values = bounded_random_walk_sequence(
    STATE_COUNT,
    low=150.0,
    high=850.0,
    start=350.0,
    step_std=18.0,
    drift=2.5,
)
bounded_random_walk_drift_integers = bounded_integer_sequence(bounded_random_walk_drift_values)

# Retain raw values for trajectory plots and integer values for tokenization
all_input_specs = {
    'sin(x)': {
        'raw': np.asarray(sin_x_sequence, dtype=float),
        'tokens': np.asarray(sin_x_integers, dtype=int),
    },
    'Gaussian switch @50%': {
        'raw': np.asarray(regime_switch_gaussian_50_sequence, dtype=float),
        'tokens': np.asarray(regime_switch_gaussian_50_integers, dtype=int),
    },
    'Gaussian switches @35%,65%': {
        'raw': np.asarray(regime_switch_gaussian_35_65_sequence, dtype=float),
        'tokens': np.asarray(regime_switch_gaussian_35_65_integers, dtype=int),
    },
    'Random walk': {
        'raw': np.asarray(bounded_random_walk_values, dtype=float),
        'tokens': np.asarray(bounded_random_walk_integers, dtype=int),
    },
    'Random walk with drift': {
        'raw': np.asarray(bounded_random_walk_drift_values, dtype=float),
        'tokens': np.asarray(bounded_random_walk_drift_integers, dtype=int),
    },
}

raw_state_space_series = {name: np.asarray(all_input_specs[name]['raw'], dtype=float) for name in DATA_ORDER}
input_data_types = {name: np.asarray(all_input_specs[name]['tokens'], dtype=int) for name in DATA_ORDER}

for name in DATA_ORDER:
    tokens = input_data_types[name]
    print(f'{name:30s} states={tokens.size:4d} token range=[{tokens.min()}, {tokens.max()}]')


Load the tokenizer, validate the separator token, and define lazy model loading.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model_hs = None
num_layers = None
hidden_size = None

# Identify the separator token used to distinguish node types
comma_token_ids = tokenizer(',', add_special_tokens=False).input_ids
if len(comma_token_ids) != 1:
    raise ValueError(f"Expected one token for ',', got {comma_token_ids}")
comma_token_id = int(comma_token_ids[0])


# Load model weights only when an archive needs to be generated
def load_attention_model(model_name=MODEL_NAME):
    global model_hs, num_layers, hidden_size
    if model_hs is None:
        model_hs = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            attn_implementation='eager',
        )
        model_hs.eval()
        num_layers = int(model_hs.config.num_hidden_layers)
        hidden_size = int(model_hs.config.hidden_size)
        print(f'Loaded {model_name}: {num_layers} layers, hidden size {hidden_size}')
    return model_hs


Define context sizing, BOS removal, hidden-state SVD, and attention-graph layout helpers.


In [ ]:
# Hidden-state signal, graph construction, and layout settings
SIGNAL_LAYER_INDEX = -2  # Hidden-state tuple index used for node-color projections
rgb_graph_layout_method = 'spring'
rgb_graph_layout_seed = 42
# Mass and uniform weighting are equivalent for the models studied here; mass also supports sparse-attention variants
rgb_graph_head_weighting = 'mass'
rgb_graph_drop_bos = True
rgb_graph_renormalize_after_drop = True
rgb_graph_layout_edge_threshold_factor = 0.0
rgb_graph_draw_edge_threshold_factor = 2.0
rgb_graph_time_init_mode = 'spiral'
rgb_graph_time_init_spring_iterations = 100
rgb_graph_time_init_pin_endpoints = False


def context_fraction_label(frac):
    pct = int(round(100 * float(frac)))
    return f'context{pct}'


def context_display_label(frac):
    pct = int(round(100 * float(frac)))
    return f'{pct}% context'


def context_state_count_from_frac(frac, total_states=STATE_COUNT):
    frac_use = float(frac)
    if frac_use >= 1.0:
        return int(total_states)
    ctx_len = max(1, int(round(frac_use * (int(total_states) - 1))))
    return min(int(total_states), ctx_len)


def safe_data_stem(data_name, frac=CONTEXT_FRACS[-1]):
    name = str(data_name).replace('=', '_').replace('.', '_')
    name = re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()
    return f'{name}_{context_fraction_label(frac)}'


def graph_data_path(data_name, frac=CONTEXT_FRACS[-1], data_dir=PROCESSED_DATA_DIR):
    return Path(data_dir) / f'{safe_data_stem(data_name, frac)}_graph_data.npz'


def build_input_sequence_from_frac(sequence, frac):
    sequence = np.asarray(sequence, dtype=int)
    if sequence.size != STATE_COUNT:
        raise ValueError(f'Expected {STATE_COUNT} states, got {sequence.size}')
    ctx_len = context_state_count_from_frac(frac, total_states=sequence.size)
    context_window = sequence[:ctx_len]
    # Encode each state as a three-digit integer followed by a comma.
    input_sequence = ','.join([f'{int(x):03d}' for x in context_window]) + ','
    return input_sequence, int(ctx_len)


def get_token_start_index(input_ids, bos_token_id=None, drop_bos=True):
    if not drop_bos:
        return 0
    if input_ids is None or bos_token_id is None:
        return 0
    if isinstance(input_ids, torch.Tensor):
        first_id = int(input_ids.flatten()[0].item())
    else:
        first_id = int(np.asarray(input_ids).flatten()[0])
    return 1 if first_id == int(bos_token_id) else 0


def extract_hidden_signal_matrix(hidden_states, input_ids, bos_token_id=None, layer_index=-2, drop_bos=True):
    token_start = get_token_start_index(input_ids, bos_token_id=bos_token_id, drop_bos=drop_bos)
    resolved_layer_index = layer_index if layer_index >= 0 else len(hidden_states) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(hidden_states):
        raise ValueError(f'Invalid layer_index={layer_index} for {len(hidden_states)} hidden-state tensors')
    signal = hidden_states[resolved_layer_index][0].float()[token_start:, :]
    return signal.cpu().numpy(), int(signal.shape[0]), int(resolved_layer_index)


def svd_projection_uncentered(signal_matrix, k=3):
    X = np.asarray(signal_matrix, dtype=np.float32)
    coords = np.zeros((X.shape[0], k), dtype=np.float32)
    energy = np.full(k, np.nan, dtype=np.float32)
    if X.ndim != 2 or X.shape[0] == 0:
        return coords, energy
    # Project the uncentered signal onto its leading right singular vectors: X V = U S.
    U, S, _ = np.linalg.svd(X, full_matrices=False)
    n_comp = min(k, S.shape[0])
    coords[:, :n_comp] = U[:, :n_comp] * S[:n_comp]
    total = np.sum(S ** 2)
    if total > 0:
        energy[:n_comp] = (S[:n_comp] ** 2) / total
    return coords, energy


def minmax_scale_rgb_channels(coords, fill_value=0.5, eps=1e-12):
    coords = np.asarray(coords, dtype=float)
    if coords.ndim == 1:
        coords = coords[:, None]
    if coords.ndim != 2:
        raise ValueError('coords must be 2D (num_tokens x num_components)')
    n = coords.shape[0]
    rgb = np.full((n, 3), float(fill_value), dtype=float)
    if n == 0:
        return rgb
    for k in range(3):
        if coords.shape[1] <= k:
            continue
        col = np.asarray(coords[:, k], dtype=float)
        finite = np.isfinite(col)
        if not np.any(finite):
            continue
        cmin = float(np.min(col[finite]))
        cmax = float(np.max(col[finite]))
        if (cmax - cmin) <= float(eps):
            rgb[finite, k] = float(fill_value)
        else:
            rgb[finite, k] = (col[finite] - cmin) / (cmax - cmin)
    rgb[~np.isfinite(rgb)] = float(fill_value)
    return np.clip(rgb, 0.0, 1.0)


def edge_weight_threshold_from_uniform(n_nodes, factor):
    n_nodes = int(n_nodes)
    if n_nodes <= 0:
        return float('inf')
    return float(factor) / float(n_nodes)


def extract_attention_graph_last_layer(outputs_attn, input_ids, bos_token_id=None, drop_bos=True, renormalize_after_drop=True, head_weighting='mass'):
    if outputs_attn.attentions is None:
        raise ValueError('Model did not return attentions. Use eager attention implementation.')
    last_layer_attn = outputs_attn.attentions[-1][0].float()
    token_start = get_token_start_index(input_ids, bos_token_id=bos_token_id, drop_bos=drop_bos)
    A = last_layer_attn[:, token_start:, token_start:]
    # Restore row sums after removing attention to the BOS token.
    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums
    # Symmetrize each attention head before combining its undirected edge weights.
    W_heads = 0.5 * (A + A.transpose(-1, -2))
    if head_weighting == 'uniform':
        alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
    elif head_weighting == 'mass':
        s = A.sum(dim=(1, 2))
        s_sum = s.sum()
        if s_sum <= 0:
            alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
        else:
            alpha = s / s_sum
    else:
        raise ValueError("head_weighting must be 'mass' or 'uniform'")
    Wbar = (alpha[:, None, None] * W_heads).sum(dim=0)
    return Wbar.detach().cpu().numpy(), int(token_start)


def build_time_ordered_init_positions(n, mode='spiral'):
    n = int(n)
    if n <= 0:
        return np.zeros((0, 2), dtype=float)
    if n == 1:
        return np.zeros((1, 2), dtype=float)
    t = np.linspace(0.0, 1.0, n)
    mode_use = str(mode).lower()
    if mode_use == 'circle':
        theta = -0.5 * np.pi + 2.0 * np.pi * t
        x = np.cos(theta)
        y = np.sin(theta)
    elif mode_use == 'spiral':
        turns = 2.25 if n < 80 else 2.75
        theta = -0.5 * np.pi + 2.0 * np.pi * turns * t
        r = 0.15 + 0.85 * t
        x = r * np.cos(theta)
        y = r * np.sin(theta)
    elif mode_use == 'line':
        x = -1.0 + 2.0 * t
        y = np.zeros_like(x)
    else:
        raise ValueError("mode must be one of: 'circle', 'spiral', 'line'")
    pos = np.column_stack([x, y]).astype(float)
    pos = pos - np.mean(pos, axis=0, keepdims=True)
    radius = np.max(np.linalg.norm(pos, axis=1))
    if np.isfinite(radius) and radius > 0:
        pos = pos / radius
    return pos


def build_topological_layout_from_attention_time_init(Wbar, init_mode='spiral', seed=None, threshold_factor=None, spring_iterations=None, pin_endpoints=False):
    W = np.asarray(Wbar, dtype=float).copy()
    if W.ndim != 2 or W.shape[0] != W.shape[1]:
        raise ValueError('Wbar must be square')
    n = W.shape[0]
    if n == 0:
        return np.zeros((0, 2), dtype=float)
    if n == 1:
        return np.zeros((1, 2), dtype=float)
    W[~np.isfinite(W)] = 0.0
    W = np.clip(W, a_min=0.0, a_max=None)
    np.fill_diagonal(W, 0.0)
    factor_use = rgb_graph_layout_edge_threshold_factor if threshold_factor is None else float(threshold_factor)
    if float(factor_use) > 0.0:
        layout_threshold = edge_weight_threshold_from_uniform(n, factor=factor_use)
        W[W <= layout_threshold] = 0.0
    # Initialize positions in token order, then relax them using attention-weighted springs.
    init_xy = build_time_ordered_init_positions(n, mode=init_mode)
    G = nx.from_numpy_array(W)
    if G.number_of_edges() == 0:
        return init_xy
    pos0 = {i: init_xy[i] for i in range(n)}
    spring_kwargs = dict(
        weight='weight',
        seed=rgb_graph_layout_seed if seed is None else int(seed),
        pos=pos0,
        iterations=max(1, int(rgb_graph_time_init_spring_iterations if spring_iterations is None else spring_iterations)),
    )
    if bool(pin_endpoints) and n > 1:
        spring_kwargs['fixed'] = [0, n - 1]
    pos_dict = nx.spring_layout(G, **spring_kwargs)
    pos = np.asarray([pos_dict[i] for i in range(n)], dtype=float)
    if pos.ndim == 1:
        pos = pos[:, None]
    if pos.shape[1] < 2:
        pos = np.column_stack([pos[:, 0], np.zeros(n)])
    pos = pos[:, :2]
    pos = pos - np.mean(pos, axis=0, keepdims=True)
    radius = np.max(np.linalg.norm(pos, axis=1))
    if np.isfinite(radius) and radius > 0:
        pos = pos / radius
    return pos

Extract last-layer attention graphs and hidden-state projections.


In [ ]:
def compute_processed_last_layer_data(data_name, frac):
    if data_name not in input_data_types:
        raise KeyError(f'Unknown data_name={data_name!r}. Available: {list(input_data_types)}')

    sequence = input_data_types[data_name]
    input_sequence, ctx_len = build_input_sequence_from_frac(sequence, frac)
    attn_model = load_attention_model(MODEL_NAME)
    attn_device = next(attn_model.parameters()).device
    inputs = tokenizer(input_sequence, return_tensors='pt').to(attn_device)
    input_ids = inputs.input_ids[0]

    print(f'Computing {data_name}: {context_display_label(frac)}, states={ctx_len}, model tokens={input_ids.shape[-1]}')
    with torch.no_grad():
        outputs = attn_model(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            return_dict=True,
        )

    signal_matrix, _seq_len, _used_hidden_layer = extract_hidden_signal_matrix(
        outputs.hidden_states,
        input_ids=input_ids,
        bos_token_id=tokenizer.bos_token_id,
        layer_index=SIGNAL_LAYER_INDEX,
        drop_bos=rgb_graph_drop_bos,
    )
    coords, _top3_energy = svd_projection_uncentered(signal_matrix, k=3)
    Wbar, token_start = extract_attention_graph_last_layer(
        outputs,
        input_ids=input_ids,
        bos_token_id=tokenizer.bos_token_id,
        drop_bos=rgb_graph_drop_bos,
        renormalize_after_drop=rgb_graph_renormalize_after_drop,
        head_weighting=rgb_graph_head_weighting,
    )

    # Align graph nodes, hidden-state coordinates, and separator labels after BOS removal.
    token_ids = input_ids[token_start:].detach().cpu().numpy().astype(np.int32)
    n_nodes = min(coords.shape[0], token_ids.shape[0], Wbar.shape[0])
    coords = np.asarray(coords[:n_nodes], dtype=np.float32)
    token_ids = np.asarray(token_ids[:n_nodes], dtype=np.int32)
    Wbar = np.asarray(Wbar[:n_nodes, :n_nodes], dtype=np.float32)
    is_delimiter = token_ids == comma_token_id
    layout_positions = build_topological_layout_from_attention_time_init(
        Wbar,
        init_mode=rgb_graph_time_init_mode,
        seed=rgb_graph_layout_seed,
        threshold_factor=rgb_graph_layout_edge_threshold_factor,
        spring_iterations=rgb_graph_time_init_spring_iterations,
        pin_endpoints=rgb_graph_time_init_pin_endpoints,
    ).astype(np.float32)

    # Keep the full raw trajectory alongside the graph for this context fraction.
    processed_data = {
        'Wbar': Wbar,
        'layout_positions': layout_positions,
        'svd_projection_coords': coords,
        'is_delimiter': np.asarray(is_delimiter, dtype=bool),
        'state_values': np.asarray(raw_state_space_series[data_name], dtype=np.float32),
    }

    del outputs, inputs, input_ids, signal_matrix
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return processed_data


Save, load, and validate the processed graph archives.


In [ ]:
def _atomic_savez_compressed(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + '.tmp')
    with tmp_path.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    tmp_path.replace(path)


def save_processed_graph_data(data_name, frac, processed_data, data_dir=PROCESSED_DATA_DIR):
    path = graph_data_path(data_name, frac=frac, data_dir=data_dir)
    missing = [key for key in GRAPH_DATA_ARRAY_KEYS if key not in processed_data]
    if missing:
        raise KeyError(f'Missing required processed arrays: {missing}')
    arrays_to_save = {key: processed_data[key] for key in GRAPH_DATA_ARRAY_KEYS}
    _atomic_savez_compressed(path, **arrays_to_save)
    return path


def load_processed_graph_data(data_name, frac, data_dir=PROCESSED_DATA_DIR):
    path = graph_data_path(data_name, frac=frac, data_dir=data_dir)
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as npz:
        processed_data = {key: npz[key] for key in npz.files}
    missing = [key for key in GRAPH_DATA_ARRAY_KEYS if key not in processed_data]
    if missing:
        raise KeyError(f'{path} is missing required arrays: {missing}')
    extra = sorted(set(processed_data) - set(GRAPH_DATA_ARRAY_KEYS))
    if extra:
        raise KeyError(f'{path} contains nonessential arrays and should be rebuilt: {extra}')
    return processed_data


def processed_graph_data_complete(data_name, frac, data_dir=PROCESSED_DATA_DIR):
    try:
        data = load_processed_graph_data(data_name, frac=frac, data_dir=data_dir)
    except (FileNotFoundError, OSError, KeyError, ValueError):
        return False

    Wbar = np.asarray(data['Wbar'])
    if Wbar.ndim != 2 or Wbar.shape[0] != Wbar.shape[1]:
        return False
    n_nodes = int(Wbar.shape[0])
    expected_nodes = 2 * context_state_count_from_frac(frac)
    if n_nodes != int(expected_nodes):
        return False

    expected_shapes = {
        'layout_positions': (n_nodes, 2),
        'svd_projection_coords': (n_nodes, 3),
        'is_delimiter': (n_nodes,),
        'state_values': (STATE_COUNT,),
    }
    return all(np.asarray(data[key]).shape == shape for key, shape in expected_shapes.items())

Generate or reuse every required input archive at the five selected context fractions.


In [ ]:
generated_paths = []
skipped_paths = []

# One archive for each input family and context fraction
generation_jobs = [
    (data_name, float(frac))
    for frac in CONTEXT_FRACS
    for data_name in DATA_ORDER
]

for data_name, frac in tqdm(generation_jobs, desc='Selected-context last-layer graph/SVD data', unit='archive'):
    output_path = graph_data_path(data_name, frac=frac)
    # Reuse archives only when their keys and array shapes match the expected schema.
    if processed_graph_data_complete(data_name, frac=frac):
        skipped_paths.append(output_path)
        continue

    if output_path.exists():
        print(f'Rebuilding incomplete or outdated archive: {output_path}')

    processed = compute_processed_last_layer_data(data_name, frac=frac)
    saved_path = save_processed_graph_data(data_name, frac, processed)
    generated_paths.append(saved_path)

print(f'Generated {len(generated_paths)} archive(s); skipped {len(skipped_paths)} existing archive(s).')
print(f'Processed graph data directory: {PROCESSED_DATA_DIR}')